In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import numpy as np
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from preprocess import gen_data_set, gen_model_input
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model

from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss, NegativeSampler
import sys

sys.path.append("../src/otto_recommender")
import utils
import deepmatch_preprocess

2023-01-22 02:45:16.958114: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-01-22 02:45:17.096093: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-01-22 02:45:17.096128: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2023-01-22 02:45:17.798830: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-

INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch13_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"
    new_split = "_new_split"


config = Config()

In [3]:
%%time

X_train_old_split, X_val_old_split, X_val_seq_old_split, ordinal_encoder = utils.get_train_val_dfs(config, new_split=False, do_ordinal_encoder_transform=False)

CPU times: user 13min 51s, sys: 1min 49s, total: 15min 40s
Wall time: 1min 47s


In [4]:
%%time

X_train_new_split, X_val_new_split, X_val_seq_new_split, ordinal_encoder = utils.get_train_val_dfs(config, new_split=True, do_ordinal_encoder_transform=False)

CPU times: user 12min 58s, sys: 14min 4s, total: 27min 2s
Wall time: 2min 4s


In [5]:
X_train_new_split.shape, X_train_old_split.shape

((221788865, 7), (220423451, 6))

In [6]:
X_val_new_split.shape, X_val_old_split.shape

((1855354, 6), (3220768, 6))

In [7]:
X_val_seq_new_split.shape, X_val_seq_old_split.shape

((5072769, 7), (3707355, 6))

In [8]:
len(X_train_new_split) + len(X_val_new_split)

223644219